# Uplift Modelling: A Hands-On Walkthrough

This notebook walks through the difference between a **response model** (traditional classification) and an **uplift model** (T-Learner) using a small dataset.

**The key question**: A response model predicts *who will engage*. An uplift model predicts *who will engage differently because of the treatment*. Let's see why that distinction matters.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

## Step 1: Create the Experimental Dataset

We ran an experiment: 12 players were **randomly** split into treatment (got a bonus) and control (no bonus). We observed whether each player engaged afterward.

Crucially, each player also has a **hidden segment** (their true type) that we wouldn't know in practice. We include it here so we can see what the models get right and wrong.

In [2]:
# Treatment group: these players received a bonus
treatment = pd.DataFrame({
    'Player': ['A', 'B', 'C', 'D', 'E', 'F'],
    'DaysSinceLogin': [2, 14, 30, 3, 20, 8],
    'MonthlySpend': [500, 100, 20, 450, 60, 200],
    'Engaged': [1, 1, 0, 1, 0, 1],
    'Group': ['treatment'] * 6,
    # Hidden truth - we wouldn't know this in practice
    'TrueSegment': ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sure Thing', 'Lost Cause', 'Persuadable']
})

# Control group: these players did NOT receive a bonus
control = pd.DataFrame({
    'Player': ['G', 'H', 'I', 'J', 'K', 'L'],
    'DaysSinceLogin': [3, 12, 28, 2, 18, 7],
    'MonthlySpend': [480, 120, 30, 550, 70, 250],
    'Engaged': [1, 0, 0, 1, 0, 0],
    'Group': ['control'] * 6,
    'TrueSegment': ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sure Thing', 'Lost Cause', 'Sleeping Dog']
})

# Combined dataset
data = pd.concat([treatment, control], ignore_index=True)

print("=== TREATMENT GROUP (got bonus) ===")
print(treatment[['Player', 'DaysSinceLogin', 'MonthlySpend', 'Engaged', 'TrueSegment']].to_string(index=False))
print()
print("=== CONTROL GROUP (no bonus) ===")
print(control[['Player', 'DaysSinceLogin', 'MonthlySpend', 'Engaged', 'TrueSegment']].to_string(index=False))

=== TREATMENT GROUP (got bonus) ===
Player  DaysSinceLogin  MonthlySpend  Engaged TrueSegment
     A               2           500        1  Sure Thing
     B              14           100        1 Persuadable
     C              30            20        0  Lost Cause
     D               3           450        1  Sure Thing
     E              20            60        0  Lost Cause
     F               8           200        1 Persuadable

=== CONTROL GROUP (no bonus) ===
Player  DaysSinceLogin  MonthlySpend  Engaged  TrueSegment
     G               3           480        1   Sure Thing
     H              12           120        0  Persuadable
     I              28            30        0   Lost Cause
     J               2           550        1   Sure Thing
     K              18            70        0   Lost Cause
     L               7           250        0 Sleeping Dog


**Notice the key patterns in the hidden segments:**

- **Sure Things** (A, D, G, J): High spend, low recency. They engage regardless of bonus. A and D engaged with bonus; G and J engaged without bonus.
- **Persuadables** (B, H, F): Moderate features. B and F engaged WITH bonus; H did NOT engage WITHOUT bonus. The bonus made the difference.
- **Lost Causes** (C, E, I, K): High recency, low spend. They don't engage regardless. C and E didn't engage even with bonus; I and K didn't engage without bonus either.
- **Sleeping Dog** (L): Moderate features but would disengage if contacted. L didn't engage without bonus (good - left alone).

## Step 2: The Response Model (Traditional Approach)

A traditional response model trains on **only the treatment group**. It learns: given a player's features, how likely are they to engage after receiving a bonus?

It has **no access to the control group** - it doesn't know what would have happened without the bonus.

In [3]:
# Prepare treatment-only data for the response model
features = ['DaysSinceLogin', 'MonthlySpend']

X_treatment = treatment[features]
y_treatment = treatment['Engaged']

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_treatment_scaled = scaler.fit_transform(X_treatment)

# Train the response model
response_model = LogisticRegression(random_state=42)
response_model.fit(X_treatment_scaled, y_treatment)

print("Response model trained on treatment group only.")
print(f"  Learned coefficients:")
print(f"  DaysSinceLogin: {response_model.coef_[0][0]:.3f} (negative = more recent login -> more likely to engage)")
print(f"  MonthlySpend:   {response_model.coef_[0][1]:.3f} (positive = higher spend -> more likely to engage)")
print(f"  Intercept:      {response_model.intercept_[0]:.3f}")

Response model trained on treatment group only.
  Learned coefficients:
  DaysSinceLogin: -0.924 (negative = more recent login -> more likely to engage)
  MonthlySpend:   0.552 (positive = higher spend -> more likely to engage)
  Intercept:      0.999


In [4]:
response_model.coef_

array([[-0.92376042,  0.55208926]])

## Step 3: The T-Learner Uplift Model

The T-Learner trains **two separate models**:
- **Model 1** on the treatment group: P(engage | features, got bonus)
- **Model 0** on the control group: P(engage | features, no bonus)

The uplift for any player is: Model 1 prediction - Model 0 prediction

In [5]:
# Prepare data for both models
X_treatment = treatment[features]
y_treatment = treatment['Engaged']

X_control = control[features]
y_control = control['Engaged']

# Use the SAME scaler for both (fit on combined data for consistency)
scaler_uplift = StandardScaler()
X_all = pd.concat([X_treatment, X_control])
scaler_uplift.fit(X_all)

X_treatment_scaled = scaler_uplift.transform(X_treatment)
X_control_scaled = scaler_uplift.transform(X_control)

# Train Model 1: P(engage | features, got bonus)
model_treatment = LogisticRegression(random_state=42)
model_treatment.fit(X_treatment_scaled, y_treatment)

# Train Model 0: P(engage | features, no bonus)
model_control = LogisticRegression(random_state=42)
model_control.fit(X_control_scaled, y_control)

print("T-Learner trained: two separate models.")
print()
print("Model 1 (treatment) coefficients:")
print(f"  DaysSinceLogin: {model_treatment.coef_[0][0]:.3f}")
print(f"  MonthlySpend:   {model_treatment.coef_[0][1]:.3f}")
print()
print("Model 0 (control) coefficients:")
print(f"  DaysSinceLogin: {model_control.coef_[0][0]:.3f}")
print(f"  MonthlySpend:   {model_control.coef_[0][1]:.3f}")

T-Learner trained: two separate models.

Model 1 (treatment) coefficients:
  DaysSinceLogin: -0.942
  MonthlySpend:   0.528

Model 0 (control) coefficients:
  DaysSinceLogin: -0.547
  MonthlySpend:   1.019


## Step 4: Score New Players and Compare

Now let's create some new players with known hidden segments and see how each approach ranks them.

In [6]:
# New players to score
new_players = pd.DataFrame({
    'Player': ['M', 'N', 'O', 'P', 'Q', 'R'],
    'DaysSinceLogin': [1, 10, 35, 4, 22, 6],
    'MonthlySpend': [600, 150, 15, 520, 50, 180],
    # Hidden truth
    'TrueSegment': ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sure Thing', 'Lost Cause', 'Persuadable']
})

X_new = new_players[features]

# --- Response Model Scores ---
X_new_scaled_resp = scaler.transform(X_new)
new_players['ResponseScore'] = response_model.predict_proba(X_new_scaled_resp)[:, 1]

# --- T-Learner Uplift Scores ---
X_new_scaled_uplift = scaler_uplift.transform(X_new)
p_treat = model_treatment.predict_proba(X_new_scaled_uplift)[:, 1]
p_control = model_control.predict_proba(X_new_scaled_uplift)[:, 1]
new_players['P(engage|bonus)'] = p_treat
new_players['P(engage|no bonus)'] = p_control
new_players['UpliftScore'] = p_treat - p_control

# Display results
print("=== SCORING RESULTS ===")
print()
display_cols = ['Player', 'DaysSinceLogin', 'MonthlySpend', 'TrueSegment',
                'ResponseScore', 'P(engage|bonus)', 'P(engage|no bonus)', 'UpliftScore']
result = new_players[display_cols].copy()
for col in ['ResponseScore', 'P(engage|bonus)', 'P(engage|no bonus)', 'UpliftScore']:
    result[col] = result[col].round(3)
print(result.to_string(index=False))

=== SCORING RESULTS ===

Player  DaysSinceLogin  MonthlySpend TrueSegment  ResponseScore  P(engage|bonus)  P(engage|no bonus)  UpliftScore
     M               1           600  Sure Thing          0.962            0.961               0.803        0.158
     N              10           150 Persuadable          0.742            0.747               0.187        0.560
     O              35            15  Lost Cause          0.157            0.147               0.026        0.121
     P               4           520  Sure Thing          0.937            0.936               0.693        0.243
     Q              22            50  Lost Cause          0.410            0.408               0.064        0.344
     R               6           180 Persuadable          0.820            0.827               0.253        0.574


## Step 5: Compare Rankings - Who Gets the Bonus?

If we have budget for only **2 bonuses**, who does each model pick?

In [7]:
budget = 2

# Response model ranking
resp_ranked = new_players.sort_values('ResponseScore', ascending=False)
resp_picks = resp_ranked.head(budget)

# Uplift model ranking
uplift_ranked = new_players.sort_values('UpliftScore', ascending=False)
uplift_picks = uplift_ranked.head(budget)

print(f"Budget: {budget} bonuses to give out")
print()
print("=== RESPONSE MODEL PICKS ===")
print(f"  Ranks by: P(engage after getting bonus)")
for _, row in resp_picks.iterrows():
    print(f"  -> Player {row['Player']}: score {row['ResponseScore']:.3f} (True segment: {row['TrueSegment']})")
print()
print("=== UPLIFT MODEL PICKS ===")
print(f"  Ranks by: P(engage|bonus) - P(engage|no bonus)")
for _, row in uplift_picks.iterrows():
    print(f"  -> Player {row['Player']}: uplift {row['UpliftScore']:.3f} (True segment: {row['TrueSegment']})")
print()
print("=== FULL RANKINGS SIDE BY SIDE ===")
comparison = pd.DataFrame({
    'ResponseRank': [f"{r['Player']} ({r['TrueSegment']}) [{r['ResponseScore']:.3f}]" 
                     for _, r in resp_ranked.iterrows()],
    'UpliftRank': [f"{r['Player']} ({r['TrueSegment']}) [{r['UpliftScore']:.3f}]" 
                   for _, r in uplift_ranked.iterrows()]
}, index=range(1, len(new_players) + 1))
comparison.index.name = 'Rank'
print(comparison.to_string())

Budget: 2 bonuses to give out

=== RESPONSE MODEL PICKS ===
  Ranks by: P(engage after getting bonus)
  -> Player M: score 0.962 (True segment: Sure Thing)
  -> Player P: score 0.937 (True segment: Sure Thing)

=== UPLIFT MODEL PICKS ===
  Ranks by: P(engage|bonus) - P(engage|no bonus)
  -> Player R: uplift 0.574 (True segment: Persuadable)
  -> Player N: uplift 0.560 (True segment: Persuadable)

=== FULL RANKINGS SIDE BY SIDE ===
                 ResponseRank               UpliftRank
Rank                                                  
1      M (Sure Thing) [0.962]  R (Persuadable) [0.574]
2      P (Sure Thing) [0.937]  N (Persuadable) [0.560]
3     R (Persuadable) [0.820]   Q (Lost Cause) [0.344]
4     N (Persuadable) [0.742]   P (Sure Thing) [0.243]
5      Q (Lost Cause) [0.410]   M (Sure Thing) [0.158]
6      O (Lost Cause) [0.157]   O (Lost Cause) [0.121]


## Step 6: Why This Matters - The Business Impact

Let's put numbers on it. Assume each bonus costs 10 and each retained player is worth 100.

In [8]:
bonus_cost = 10
player_value = 100

# What happens with the response model's picks?
print("=== RESPONSE MODEL OUTCOME ===")
resp_total = 0
for _, row in resp_picks.iterrows():
    segment = row['TrueSegment']
    if segment == 'Sure Thing':
        incremental = 0  # Would have engaged anyway
        note = "would have engaged anyway - bonus wasted"
    elif segment == 'Persuadable':
        incremental = player_value  # Bonus caused engagement
        note = "bonus caused engagement - good spend"
    elif segment == 'Lost Cause':
        incremental = 0  # Won't engage regardless
        note = "won't engage regardless - bonus wasted"
    elif segment == 'Sleeping Dog':
        incremental = -player_value  # Bonus caused disengagement
        note = "bonus caused DISENGAGEMENT - harmful"
    
    profit = incremental - bonus_cost
    resp_total += profit
    print(f"  Player {row['Player']} ({segment}): {note}")
    print(f"    Incremental value: {incremental}, Cost: {bonus_cost}, Profit: {profit}")
print(f"  TOTAL PROFIT: {resp_total}")

print()

# What happens with the uplift model's picks?
print("=== UPLIFT MODEL OUTCOME ===")
uplift_total = 0
for _, row in uplift_picks.iterrows():
    segment = row['TrueSegment']
    if segment == 'Sure Thing':
        incremental = 0
        note = "would have engaged anyway - bonus wasted"
    elif segment == 'Persuadable':
        incremental = player_value
        note = "bonus caused engagement - good spend"
    elif segment == 'Lost Cause':
        incremental = 0
        note = "won't engage regardless - bonus wasted"
    elif segment == 'Sleeping Dog':
        incremental = -player_value
        note = "bonus caused DISENGAGEMENT - harmful"
    
    profit = incremental - bonus_cost
    uplift_total += profit
    print(f"  Player {row['Player']} ({segment}): {note}")
    print(f"    Incremental value: {incremental}, Cost: {bonus_cost}, Profit: {profit}")
print(f"  TOTAL PROFIT: {uplift_total}")

print()
print(f"=== DIFFERENCE: Uplift model profit ({uplift_total}) vs Response model profit ({resp_total}) ===")

=== RESPONSE MODEL OUTCOME ===
  Player M (Sure Thing): would have engaged anyway - bonus wasted
    Incremental value: 0, Cost: 10, Profit: -10
  Player P (Sure Thing): would have engaged anyway - bonus wasted
    Incremental value: 0, Cost: 10, Profit: -10
  TOTAL PROFIT: -20

=== UPLIFT MODEL OUTCOME ===
  Player R (Persuadable): bonus caused engagement - good spend
    Incremental value: 100, Cost: 10, Profit: 90
  Player N (Persuadable): bonus caused engagement - good spend
    Incremental value: 100, Cost: 10, Profit: 90
  TOTAL PROFIT: 180

=== DIFFERENCE: Uplift model profit (180) vs Response model profit (-20) ===


## Step 7: Larger Dataset - See the Pattern Clearly

With only 12 training players, the models are noisy. Let's generate a larger dataset (200 players) with known segments so the pattern is unmistakable.

In [9]:
np.random.seed(42)
n = 200

# Assign true segments
segments = np.random.choice(
    ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sleeping Dog'],
    size=n,
    p=[0.40, 0.15, 0.35, 0.10]  # Same proportions as the paper
)

# Generate features based on segment (so features are informative)
days_since_login = np.zeros(n)
monthly_spend = np.zeros(n)

for i, seg in enumerate(segments):
    if seg == 'Sure Thing':
        days_since_login[i] = np.random.uniform(1, 8)
        monthly_spend[i] = np.random.uniform(300, 700)
    elif seg == 'Persuadable':
        days_since_login[i] = np.random.uniform(5, 20)
        monthly_spend[i] = np.random.uniform(80, 250)
    elif seg == 'Lost Cause':
        days_since_login[i] = np.random.uniform(20, 50)
        monthly_spend[i] = np.random.uniform(5, 60)
    elif seg == 'Sleeping Dog':
        days_since_login[i] = np.random.uniform(3, 15)
        monthly_spend[i] = np.random.uniform(150, 400)

# Add noise to features
days_since_login += np.random.normal(0, 2, n)
days_since_login = np.clip(days_since_login, 1, 60)
monthly_spend += np.random.normal(0, 30, n)
monthly_spend = np.clip(monthly_spend, 0, 1000)

# Randomly assign to treatment/control (50/50)
treatment_flag = np.random.binomial(1, 0.5, n)

# Generate outcomes based on segment and treatment
# These are the TRUE causal response probabilities
engaged = np.zeros(n, dtype=int)
for i in range(n):
    seg = segments[i]
    treated = treatment_flag[i]
    
    if seg == 'Sure Thing':
        p = 0.85 if treated else 0.82  # Engages either way
    elif seg == 'Persuadable':
        p = 0.70 if treated else 0.30  # BIG difference with treatment
    elif seg == 'Lost Cause':
        p = 0.10 if treated else 0.08  # Low either way
    elif seg == 'Sleeping Dog':
        p = 0.45 if treated else 0.65  # WORSE with treatment
    
    engaged[i] = np.random.binomial(1, p)

# Build dataframe
big_data = pd.DataFrame({
    'DaysSinceLogin': days_since_login.round(1),
    'MonthlySpend': monthly_spend.round(0),
    'Treatment': treatment_flag,
    'Engaged': engaged,
    'TrueSegment': segments
})

# Show segment distribution and engagement rates
print("=== DATASET SUMMARY ===")
print(f"Total players: {n}")
print(f"Treatment: {treatment_flag.sum()}, Control: {n - treatment_flag.sum()}")
print()

summary = big_data.groupby('TrueSegment').agg(
    Count=('Engaged', 'size'),
    Treated=('Treatment', 'sum'),
    EngageRate_Treated=('Engaged', lambda x: x[big_data.loc[x.index, 'Treatment'] == 1].mean()),
    EngageRate_Control=('Engaged', lambda x: x[big_data.loc[x.index, 'Treatment'] == 0].mean())
).round(3)

# Cleaner summary
for seg in ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sleeping Dog']:
    seg_data = big_data[big_data['TrueSegment'] == seg]
    treated = seg_data[seg_data['Treatment'] == 1]
    control = seg_data[seg_data['Treatment'] == 0]
    print(f"{seg}:")
    print(f"  Count: {len(seg_data)} (treated: {len(treated)}, control: {len(control)})")
    t_rate = treated['Engaged'].mean() if len(treated) > 0 else 0
    c_rate = control['Engaged'].mean() if len(control) > 0 else 0
    print(f"  Engagement rate: {t_rate:.1%} (treated) vs {c_rate:.1%} (control)")
    print(f"  True uplift: {t_rate - c_rate:+.1%}")
    print()

=== DATASET SUMMARY ===
Total players: 200
Treatment: 104, Control: 96

Sure Thing:
  Count: 90 (treated: 45, control: 45)
  Engagement rate: 80.0% (treated) vs 84.4% (control)
  True uplift: -4.4%

Persuadable:
  Count: 26 (treated: 16, control: 10)
  Engagement rate: 56.2% (treated) vs 50.0% (control)
  True uplift: +6.2%

Lost Cause:
  Count: 65 (treated: 31, control: 34)
  Engagement rate: 6.5% (treated) vs 8.8% (control)
  True uplift: -2.4%

Sleeping Dog:
  Count: 19 (treated: 12, control: 7)
  Engagement rate: 50.0% (treated) vs 28.6% (control)
  True uplift: +21.4%



In [10]:
# Split into treatment and control
treat_df = big_data[big_data['Treatment'] == 1].copy()
ctrl_df = big_data[big_data['Treatment'] == 0].copy()

feats = ['DaysSinceLogin', 'MonthlySpend']

# Scale features
scaler_big = StandardScaler()
scaler_big.fit(big_data[feats])

# --- RESPONSE MODEL: trained on treatment group only ---
resp_model_big = LogisticRegression(random_state=42)
resp_model_big.fit(scaler_big.transform(treat_df[feats]), treat_df['Engaged'])

# --- T-LEARNER: two separate models ---
model_t = LogisticRegression(random_state=42)
model_t.fit(scaler_big.transform(treat_df[feats]), treat_df['Engaged'])

model_c = LogisticRegression(random_state=42)
model_c.fit(scaler_big.transform(ctrl_df[feats]), ctrl_df['Engaged'])

# Score ALL players
X_all_scaled = scaler_big.transform(big_data[feats])
big_data['ResponseScore'] = resp_model_big.predict_proba(X_all_scaled)[:, 1]
big_data['P_treat'] = model_t.predict_proba(X_all_scaled)[:, 1]
big_data['P_control'] = model_c.predict_proba(X_all_scaled)[:, 1]
big_data['UpliftScore'] = big_data['P_treat'] - big_data['P_control']

print("Models trained. Now let's see who each model targets...")

Models trained. Now let's see who each model targets...


In [11]:
# Simulate targeting: pick the top 30 players (15% of 200)
budget_big = 30

# Response model picks
resp_top = big_data.nlargest(budget_big, 'ResponseScore')
resp_segments = resp_top['TrueSegment'].value_counts()

# Uplift model picks
uplift_top = big_data.nlargest(budget_big, 'UpliftScore')
uplift_segments = uplift_top['TrueSegment'].value_counts()

print(f"Budget: target top {budget_big} players out of {n}")
print()
print("=== WHO DOES EACH MODEL TARGET? ===")
print()

all_segs = ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sleeping Dog']
comparison_df = pd.DataFrame({
    'Segment': all_segs,
    'ResponseModel': [resp_segments.get(s, 0) for s in all_segs],
    'UpliftModel': [uplift_segments.get(s, 0) for s in all_segs]
})
print(comparison_df.to_string(index=False))
print()
print("--- Interpretation ---")
print(f"Response model targets {resp_segments.get('Sure Thing', 0)} Sure Things (wasted bonus!)")
print(f"Response model targets {resp_segments.get('Persuadable', 0)} Persuadables (good)")
print(f"Response model targets {resp_segments.get('Sleeping Dog', 0)} Sleeping Dogs (harmful!)")
print()
print(f"Uplift model targets {uplift_segments.get('Sure Thing', 0)} Sure Things")
print(f"Uplift model targets {uplift_segments.get('Persuadable', 0)} Persuadables (good)")
print(f"Uplift model targets {uplift_segments.get('Sleeping Dog', 0)} Sleeping Dogs")

Budget: target top 30 players out of 200

=== WHO DOES EACH MODEL TARGET? ===

     Segment  ResponseModel  UpliftModel
  Sure Thing             30            6
 Persuadable              0           15
  Lost Cause              0            1
Sleeping Dog              0            8

--- Interpretation ---
Response model targets 30 Sure Things (wasted bonus!)
Response model targets 0 Persuadables (good)
Response model targets 0 Sleeping Dogs (harmful!)

Uplift model targets 6 Sure Things
Uplift model targets 15 Persuadables (good)
Uplift model targets 8 Sleeping Dogs


In [12]:
# Calculate the business impact
bonus_cost = 10
player_value = 100

def calculate_profit(selected_players):
    """Calculate profit from targeting a set of players."""
    total_incremental = 0
    total_cost = 0
    for _, row in selected_players.iterrows():
        total_cost += bonus_cost
        seg = row['TrueSegment']
        if seg == 'Sure Thing':
            total_incremental += 0        # No incremental value
        elif seg == 'Persuadable':
            total_incremental += player_value  # Genuine incremental value
        elif seg == 'Lost Cause':
            total_incremental += 0        # No incremental value
        elif seg == 'Sleeping Dog':
            total_incremental -= player_value  # Negative value!
    return total_incremental - total_cost

resp_profit = calculate_profit(resp_top)
uplift_profit = calculate_profit(uplift_top)
random_picks = big_data.sample(budget_big, random_state=42)
random_profit = calculate_profit(random_picks)

print("=== BUSINESS IMPACT ===")
print(f"  Random targeting profit:   ${random_profit:,}")
print(f"  Response model profit:     ${resp_profit:,}")
print(f"  Uplift model profit:       ${uplift_profit:,}")
print()
if uplift_profit > resp_profit:
    print(f"Uplift model generates ${uplift_profit - resp_profit:,} MORE profit than the response model.")
    print("This is because it targets Persuadables instead of wasting budget on Sure Things.")

=== BUSINESS IMPACT ===
  Random targeting profit:   $-200
  Response model profit:     $-300
  Uplift model profit:       $400

Uplift model generates $700 MORE profit than the response model.
This is because it targets Persuadables instead of wasting budget on Sure Things.


## Summary So Far

| Approach | What it predicts | Trained on | Limitation |
|----------|-----------------|------------|------------|
| Response Model | P(engage after bonus) | Treatment group only | Cannot distinguish Sure Things from Persuadables |
| T-Learner Uplift | P(engage with bonus) - P(engage without bonus) | Both groups separately | Needs experimental data with a control group |

The response model picks players most likely to engage - which are usually the **Sure Things** who would have engaged anyway. The uplift model picks players whose engagement **changes the most** because of the bonus - the **Persuadables**.

---

## Step 8: The Noise Problem - Why Uplift Estimates Are Unreliable

The T-Learner subtracts two model predictions. Each prediction has error. When you subtract two noisy numbers, the noise adds up. This section shows what that looks like in practice.

In [13]:
# Let's make the treatment effect SMALL (3pp) while keeping high baseline variation.
# This is realistic - most marketing interventions have small effects.
# We use a SMALL sample (100 players) to make the noise problem obvious.

np.random.seed(42)
n_noise = 100  # Small sample = noisy estimates

# Generate features
days = np.random.uniform(1, 50, n_noise)
spend = np.random.uniform(10, 700, n_noise)

# Random treatment assignment
treated_flag = np.random.binomial(1, 0.5, n_noise)

# TRUTH: the real uplift is only 3pp for everyone (small, uniform effect)
# But baseline engagement varies hugely (20% to 80%) depending on features
baseline_prob = 0.2 + 0.6 * (1 - days / 50)  # Recent players have higher baseline
true_uplift = 0.03  # Only 3 percentage points - small!

# Generate outcomes
prob_with_treatment = np.clip(baseline_prob + true_uplift, 0, 1)
prob_without = baseline_prob

actual_prob = np.where(treated_flag, prob_with_treatment, prob_without)
engaged_noise = np.random.binomial(1, actual_prob)

noise_data = pd.DataFrame({
    'DaysSinceLogin': days.round(1),
    'MonthlySpend': spend.round(0),
    'Treatment': treated_flag,
    'Engaged': engaged_noise
})

print("=== THE NOISE PROBLEM ===")
print()
print(f"TRUE uplift for every player: {true_uplift:.0%} (3 percentage points)")
print(f"Baseline engagement range: {baseline_prob.min():.0%} to {baseline_prob.max():.0%}")
print(f"So the signal (3pp) is TINY compared to the baseline variation (60pp range)")
print(f"Sample size: only {n_noise} players ({treated_flag.sum()} treated, {n_noise - treated_flag.sum()} control)")
print()

# Train T-Learner on this small noisy dataset
from sklearn.tree import DecisionTreeClassifier  # Trees overfit more = more noise

treat_noise = noise_data[noise_data['Treatment'] == 1]
ctrl_noise = noise_data[noise_data['Treatment'] == 0]

# Use decision trees - they overfit on small data, making noise more visible
m1_noise = DecisionTreeClassifier(max_depth=4, random_state=42)
m1_noise.fit(treat_noise[['DaysSinceLogin', 'MonthlySpend']], treat_noise['Engaged'])

m0_noise = DecisionTreeClassifier(max_depth=4, random_state=42)
m0_noise.fit(ctrl_noise[['DaysSinceLogin', 'MonthlySpend']], ctrl_noise['Engaged'])

# Score everyone
X_all_noise = noise_data[['DaysSinceLogin', 'MonthlySpend']]
noise_data['P_treat'] = m1_noise.predict_proba(X_all_noise)[:, 1]
noise_data['P_control'] = m0_noise.predict_proba(X_all_noise)[:, 1]
noise_data['EstimatedUplift'] = noise_data['P_treat'] - noise_data['P_control']
noise_data['TrueUplift'] = true_uplift

print(f"Estimated uplift range: {noise_data['EstimatedUplift'].min():.3f} to {noise_data['EstimatedUplift'].max():.3f}")
print(f"Estimated uplift mean:  {noise_data['EstimatedUplift'].mean():.3f}")
print(f"Estimated uplift std:   {noise_data['EstimatedUplift'].std():.3f}")
print()
print("The true uplift is 0.030 for EVERYONE.")
print(f"But estimates range from {noise_data['EstimatedUplift'].min():.3f} to {noise_data['EstimatedUplift'].max():.3f}!")
print("Some players look like they have LARGE NEGATIVE uplift. Some look 10x higher than reality.")
print("This is pure noise from subtracting two imperfect models trained on small data.")

=== THE NOISE PROBLEM ===

TRUE uplift for every player: 3% (3 percentage points)
Baseline engagement range: 21% to 78%
So the signal (3pp) is TINY compared to the baseline variation (60pp range)
Sample size: only 100 players (58 treated, 42 control)

Estimated uplift range: -1.000 to 1.000
Estimated uplift mean:  0.085
Estimated uplift std:   0.462

The true uplift is 0.030 for EVERYONE.
But estimates range from -1.000 to 1.000!
Some players look like they have LARGE NEGATIVE uplift. Some look 10x higher than reality.
This is pure noise from subtracting two imperfect models trained on small data.


### Visualising the noise

Every player has the same true uplift (3pp). But the estimated uplift is all over the place. Let's look at 10 example players to see how wrong the estimates can be.

In [14]:
# Show 10 example players - sorted by estimated uplift to show the spread
sample = noise_data.nlargest(5, 'EstimatedUplift')[['DaysSinceLogin', 'MonthlySpend', 'P_treat', 'P_control', 'EstimatedUplift', 'TrueUplift']].copy()
sample = pd.concat([sample, noise_data.nsmallest(5, 'EstimatedUplift')[['DaysSinceLogin', 'MonthlySpend', 'P_treat', 'P_control', 'EstimatedUplift', 'TrueUplift']]])

sample = sample.round(3)
sample['Error'] = (sample['EstimatedUplift'] - sample['TrueUplift']).round(3)
sample = sample.reset_index(drop=True)
sample.index = ['Top 1', 'Top 2', 'Top 3', 'Top 4', 'Top 5', 
                'Bottom 1', 'Bottom 2', 'Bottom 3', 'Bottom 4', 'Bottom 5']

print("=== 10 EXAMPLE PLAYERS: Top 5 and Bottom 5 by Estimated Uplift ===")
print()
print(sample.to_string())
print()
print("Every single player has TRUE uplift of 0.030.")
print("But the model thinks the top players have much higher uplift")
print("and the bottom players have lower (or negative!) uplift.")
print("If we target based on these estimates, we're just chasing noise.")

=== 10 EXAMPLE PLAYERS: Top 5 and Bottom 5 by Estimated Uplift ===

          DaysSinceLogin  MonthlySpend  P_treat  P_control  EstimatedUplift  TrueUplift  Error
Top 1                8.6         636.0     1.00      0.000            1.000        0.03  0.970
Top 2                7.8         628.0     1.00      0.000            1.000        0.03  0.970
Top 3                7.0         674.0     1.00      0.000            1.000        0.03  0.970
Top 4               10.7          21.0     1.00      0.000            1.000        0.03  0.970
Top 5                6.7         105.0     1.00      0.000            1.000        0.03  0.970
Bottom 1             3.2         174.0     0.00      1.000           -1.000        0.03 -1.030
Bottom 2            16.2         188.0     0.00      0.714           -0.714        0.03 -0.744
Bottom 3            35.7          63.0     0.25      0.714           -0.464        0.03 -0.494
Bottom 4            30.0          15.0     0.25      0.714           -0.464  

### Repeat the experiment: see how estimates change with different random data

If the estimates were reliable, they'd be similar each time. Let's retrain 10 times with different random splits and see how much the uplift estimate for the SAME player changes.

In [15]:
# Pick 5 fixed players to track across 10 different experiments
track_players = pd.DataFrame({
    'DaysSinceLogin': [5.0, 15.0, 25.0, 35.0, 45.0],
    'MonthlySpend': [400.0, 300.0, 200.0, 100.0, 50.0],
    'Label': ['Recent/High', 'Moderate/Mod', 'Middle', 'Lapsed/Low', 'Very Lapsed']
})

n_experiments = 10
all_estimates = {label: [] for label in track_players['Label']}

for seed in range(n_experiments):
    rng = np.random.RandomState(seed)
    
    # Generate fresh small dataset each time (same true uplift = 3pp)
    d = rng.uniform(1, 50, 100)  # Small sample!
    s = rng.uniform(10, 700, 100)
    t = rng.binomial(1, 0.5, 100)
    bp = 0.2 + 0.6 * (1 - d / 50)
    p_t = np.clip(bp + 0.03, 0, 1)
    p_c = bp
    actual_p = np.where(t, p_t, p_c)
    y = rng.binomial(1, actual_p)
    
    df = pd.DataFrame({'DaysSinceLogin': d, 'MonthlySpend': s, 'Treatment': t, 'Engaged': y})
    
    # Train T-Learner with decision trees (overfits more = more visible noise)
    tr = df[df['Treatment'] == 1]
    cr = df[df['Treatment'] == 0]
    
    m1 = DecisionTreeClassifier(max_depth=4, random_state=42)
    m1.fit(tr[['DaysSinceLogin', 'MonthlySpend']], tr['Engaged'])
    m0 = DecisionTreeClassifier(max_depth=4, random_state=42)
    m0.fit(cr[['DaysSinceLogin', 'MonthlySpend']], cr['Engaged'])
    
    # Score our tracked players
    X_track = track_players[['DaysSinceLogin', 'MonthlySpend']]
    uplift_est = m1.predict_proba(X_track)[:, 1] - m0.predict_proba(X_track)[:, 1]
    
    for j, label in enumerate(track_players['Label']):
        all_estimates[label].append(round(uplift_est[j], 3))

# Display results
print("=== SAME 5 PLAYERS, 10 DIFFERENT EXPERIMENTS ===")
print(f"True uplift for all players: 0.030 (3 percentage points)")
print()
results_df = pd.DataFrame(all_estimates)
results_df.index = [f"Experiment {i+1}" for i in range(n_experiments)]
results_df.loc['--------'] = ['--------'] * 5
results_df.loc['Mean'] = pd.to_numeric(pd.DataFrame(all_estimates).mean()).round(3)
results_df.loc['Std Dev'] = pd.to_numeric(pd.DataFrame(all_estimates).std()).round(3)
results_df.loc['TRUE'] = 0.030
print(results_df.to_string())
print()
print("Notice how wildly the estimates swing from experiment to experiment.")
print("In Experiment 1, the 'Recent/High' player might look like +0.40 uplift.")
print("In Experiment 2, the SAME player might look like -0.10.")
print("The TRUE value is always 0.030. Everything else is estimation noise.")

=== SAME 5 PLAYERS, 10 DIFFERENT EXPERIMENTS ===
True uplift for all players: 0.030 (3 percentage points)

              Recent/High Moderate/Mod    Middle Lapsed/Low Very Lapsed
Experiment 1        0.667        0.667    -0.111      0.308         0.0
Experiment 2          0.0       -0.833    -0.105     -0.105         1.0
Experiment 3          0.2       -0.467      0.25        0.0        -0.5
Experiment 4          1.0        0.333     0.318      0.818      -0.182
Experiment 5       -0.143         0.19     0.333       -0.2        -0.2
Experiment 6        0.367       -0.433    -0.633      0.051      -0.333
Experiment 7        0.357        0.357    -0.099      0.188        -0.5
Experiment 8       -0.619       -0.063    -0.063      0.556         0.0
Experiment 9          0.0          1.0    -0.842        0.0         0.0
Experiment 10         0.0          0.0       0.0      0.429         0.0
--------         --------     --------  --------   --------    --------
Mean                0.183    

## Step 9: Regularisation - Shrinking Noisy Estimates Toward Zero

The idea is simple: **if we're not confident in an estimate, pull it toward zero** rather than acting on noise.

This is Bayesian shrinkage. We say: "My prior belief is that the uplift is probably small (near zero). I'll update that belief based on the data, but only to the extent that the data is reliable."

The formula is:

**shrunk_uplift = weight x estimated_uplift**

where **weight = prior_variance / (prior_variance + estimation_variance)**

- When estimation variance is LOW (reliable estimate): weight is close to 1, so we trust the estimate
- When estimation variance is HIGH (noisy estimate): weight is close to 0, so we shrink toward zero

In [16]:
# Let's demonstrate shrinkage on our noisy data.
# We'll simulate having different levels of confidence for different players.

# Use the noise_data from Step 8 (1000 players, true uplift = 3pp for all)
# The estimated uplift ranges widely due to noise.

# Simulate estimation variance: higher for extreme players (very recent or very lapsed)
# This mimics reality - we have less data at the extremes
days_centered = (noise_data['DaysSinceLogin'] - 25) / 25  # -1 to 1, centered
estimation_variance = 0.01 + 0.04 * days_centered**2  # Higher at extremes

# Prior variance (our belief about how spread out uplift could be)
prior_variance = 0.02

# Shrinkage weight
weight = prior_variance / (prior_variance + estimation_variance)

# Apply shrinkage
noise_data['ShrunkUplift'] = weight * noise_data['EstimatedUplift']
noise_data['Weight'] = weight.round(3)

# Show the effect
print("=== SHRINKAGE IN ACTION ===")
print()
print("Players near the middle of the recency range (DaysSinceLogin ~ 25):")
print("  -> More data available, lower estimation variance, weight closer to 1")
print("  -> Estimate is trusted more")
print()
print("Players at the extremes (very recent or very lapsed):")
print("  -> Less data, higher estimation variance, weight closer to 0")  
print("  -> Estimate is shrunk toward zero")
print()

# Show examples at different recency levels
bins = [(1, 10, 'Very Recent'), (10, 20, 'Moderate'), (20, 30, 'Middle'), 
        (30, 40, 'Lapsed'), (40, 50, 'Very Lapsed')]

rows = []
for lo, hi, label in bins:
    mask = (noise_data['DaysSinceLogin'] >= lo) & (noise_data['DaysSinceLogin'] < hi)
    subset = noise_data[mask]
    rows.append({
        'Player Type': label,
        'DaysSinceLogin': f'{lo}-{hi}',
        'Count': len(subset),
        'Mean Raw Uplift': round(subset['EstimatedUplift'].mean(), 4),
        'Mean Shrink Weight': round(subset['Weight'].mean(), 3),
        'Mean Shrunk Uplift': round(subset['ShrunkUplift'].mean(), 4),
        'True Uplift': 0.030
    })

shrink_table = pd.DataFrame(rows)
print(shrink_table.to_string(index=False))
print()
print("Key insight: shrinkage pulls extreme estimates back toward zero.")
print("The raw estimates at the extremes are noisy and unreliable.")
print("After shrinkage, all estimates are closer to the true value of 0.030.")

=== SHRINKAGE IN ACTION ===

Players near the middle of the recency range (DaysSinceLogin ~ 25):
  -> More data available, lower estimation variance, weight closer to 1
  -> Estimate is trusted more

Players at the extremes (very recent or very lapsed):
  -> Less data, higher estimation variance, weight closer to 0
  -> Estimate is shrunk toward zero

Player Type DaysSinceLogin  Count  Mean Raw Uplift  Mean Shrink Weight  Mean Shrunk Uplift  True Uplift
Very Recent           1-10     23           0.3569               0.370              0.1427         0.03
   Moderate          10-20     22           0.3090               0.545              0.1492         0.03
     Middle          20-30     15           0.0195               0.659              0.0118         0.03
     Lapsed          30-40     22          -0.2030               0.550             -0.1119         0.03
Very Lapsed          40-50     18          -0.1308               0.363             -0.0524         0.03

Key insight: shrinkag

### The targeting impact of shrinkage

Now let's go back to a scenario with REAL heterogeneous effects (some players are Persuadable, some are Sure Things, etc.) and see how shrinkage changes targeting decisions.

In [17]:
# Create a dataset with heterogeneous effects AND noisy estimation
# Some players have genuine high uplift, others have noisy estimates that LOOK high

np.random.seed(99)
n_reg = 500

segments_reg = np.random.choice(
    ['Sure Thing', 'Persuadable', 'Lost Cause', 'Sleeping Dog'],
    size=n_reg, p=[0.40, 0.15, 0.35, 0.10]
)

days_reg = np.zeros(n_reg)
spend_reg = np.zeros(n_reg)
true_uplift_reg = np.zeros(n_reg)

for i, seg in enumerate(segments_reg):
    if seg == 'Sure Thing':
        days_reg[i] = np.random.uniform(1, 8)
        spend_reg[i] = np.random.uniform(300, 700)
        true_uplift_reg[i] = 0.03  # Tiny effect
    elif seg == 'Persuadable':
        days_reg[i] = np.random.uniform(5, 20)
        spend_reg[i] = np.random.uniform(80, 250)
        true_uplift_reg[i] = 0.35  # Large effect
    elif seg == 'Lost Cause':
        days_reg[i] = np.random.uniform(20, 50)
        spend_reg[i] = np.random.uniform(5, 60)
        true_uplift_reg[i] = 0.02  # Tiny effect
    elif seg == 'Sleeping Dog':
        days_reg[i] = np.random.uniform(3, 15)
        spend_reg[i] = np.random.uniform(150, 400)
        true_uplift_reg[i] = -0.20  # Negative effect

# Add feature noise
days_reg += np.random.normal(0, 3, n_reg)
days_reg = np.clip(days_reg, 1, 60)
spend_reg += np.random.normal(0, 40, n_reg)
spend_reg = np.clip(spend_reg, 0, 1000)

# Simulate noisy uplift estimates (true uplift + estimation noise)
# Key: noise is MUCH HIGHER for players at the extremes and for rare segments
# This mimics real-world estimation: fewer similar players = noisier estimates
noise_level = 0.10 + 0.20 * np.abs(days_reg - 15) / 40  # Higher noise away from moderate recency
estimated_uplift_reg = true_uplift_reg + np.random.normal(0, noise_level)

# Build the comparison
reg_data = pd.DataFrame({
    'DaysSinceLogin': days_reg.round(1),
    'MonthlySpend': spend_reg.round(0),
    'TrueSegment': segments_reg,
    'TrueUplift': true_uplift_reg,
    'EstimatedUplift': estimated_uplift_reg.round(4),
    'NoiseLevel': noise_level.round(3)
})

# Apply shrinkage
prior_var = 0.04  # Our prior on how variable uplift can be
est_var = noise_level**2  # Estimation variance (squared noise)
shrinkage_weight = prior_var / (prior_var + est_var)

reg_data['ShrunkUplift'] = (shrinkage_weight * reg_data['EstimatedUplift']).round(4)
reg_data['Weight'] = shrinkage_weight.round(3)

# Target top 50 players under each strategy
budget_reg = 50

# Raw uplift targeting
raw_top = reg_data.nlargest(budget_reg, 'EstimatedUplift')
raw_segments = raw_top['TrueSegment'].value_counts()

# Shrunk uplift targeting
shrunk_top = reg_data.nlargest(budget_reg, 'ShrunkUplift')
shrunk_segments = shrunk_top['TrueSegment'].value_counts()

print(f"=== TARGETING COMPARISON: Raw vs Shrunk Uplift (top {budget_reg} of {n_reg}) ===")
print()
all_segs = ['Persuadable', 'Sure Thing', 'Lost Cause', 'Sleeping Dog']
comp = pd.DataFrame({
    'Segment': all_segs,
    'Raw Uplift Picks': [raw_segments.get(s, 0) for s in all_segs],
    'Shrunk Uplift Picks': [shrunk_segments.get(s, 0) for s in all_segs],
    'True Uplift': ['+35pp', '+3pp', '+2pp', '-20pp']
})
print(comp.to_string(index=False))
print()

# Show why: some "raw top" picks are noise
print("=== TOP 10 BY RAW UPLIFT (some are noise!) ===")
raw_top_sorted = raw_top.sort_values('EstimatedUplift', ascending=False).head(10)
display_raw = raw_top_sorted[['TrueSegment', 'TrueUplift', 'EstimatedUplift', 'NoiseLevel', 'Weight', 'ShrunkUplift']].copy()
print(display_raw.to_string())
print()
print("Look at the 'NoiseLevel' column for non-Persuadable players.")
print("High noise + lucky draw = inflated estimate that fools the raw ranking.")
print("The 'Weight' column shows how much shrinkage trusts each estimate.")
print("Low weight = high noise = estimate gets pulled toward zero.")

=== TARGETING COMPARISON: Raw vs Shrunk Uplift (top 50 of 500) ===

     Segment  Raw Uplift Picks  Shrunk Uplift Picks True Uplift
 Persuadable                39                   47       +35pp
  Sure Thing                 2                    2        +3pp
  Lost Cause                 9                    1        +2pp
Sleeping Dog                 0                    0       -20pp

=== TOP 10 BY RAW UPLIFT (some are noise!) ===
     TrueSegment  TrueUplift  EstimatedUplift  NoiseLevel  Weight  ShrunkUplift
248  Persuadable        0.35           0.8424       0.144   0.660        0.5558
451  Persuadable        0.35           0.6744       0.115   0.752        0.5074
279   Lost Cause        0.02           0.6618       0.191   0.522        0.3454
409  Persuadable        0.35           0.6346       0.124   0.724        0.4594
474  Persuadable        0.35           0.6210       0.134   0.690        0.4282
229  Persuadable        0.35           0.5630       0.146   0.654        0.3681
404 

In [18]:
# Business impact comparison
bonus_cost = 10
player_value = 100

def calc_profit_reg(selected):
    total = 0
    for _, row in selected.iterrows():
        seg = row['TrueSegment']
        if seg == 'Persuadable':
            total += player_value
        elif seg == 'Sleeping Dog':
            total -= player_value
        # Sure Thing and Lost Cause: 0 incremental value
        total -= bonus_cost
    return total

raw_profit = calc_profit_reg(raw_top)
shrunk_profit = calc_profit_reg(shrunk_top)

print("=== BUSINESS IMPACT OF REGULARISATION ===")
print()
print(f"  Raw uplift targeting profit:    ${raw_profit:,}")
print(f"  Shrunk uplift targeting profit:  ${shrunk_profit:,}")
print()
if shrunk_profit > raw_profit:
    print(f"  Regularisation improves profit by ${shrunk_profit - raw_profit:,}")
    print()
    print("  Why? Without shrinkage, some noisy players sneak into the top ranks")
    print("  because they got lucky estimates. With shrinkage, those noisy estimates")
    print("  are pulled toward zero, and genuinely persuadable players (with")
    print("  consistent, reliable high estimates) rise to the top instead.")

=== BUSINESS IMPACT OF REGULARISATION ===

  Raw uplift targeting profit:    $3,400
  Shrunk uplift targeting profit:  $4,200

  Regularisation improves profit by $800

  Why? Without shrinkage, some noisy players sneak into the top ranks
  because they got lucky estimates. With shrinkage, those noisy estimates
  are pulled toward zero, and genuinely persuadable players (with
  consistent, reliable high estimates) rise to the top instead.


## Full Summary

| Problem | What happens | Solution |
|---------|-------------|----------|
| Response model can't see causation | Targets Sure Things (wasted budget) and Sleeping Dogs (harmful) | Use uplift model (T-Learner) with treatment AND control data |
| Uplift estimates are noisy | Two models subtracted = double the error. Small effects get drowned in noise | Regularisation / Bayesian shrinkage |
| Noise is worse at the extremes | Very recent or very lapsed players have fewer similar peers, so estimates are less reliable | Shrinkage weight varies by confidence - trust estimates more where data is abundant |

**The chain of logic:**
1. We need causal effects, not just predictions -> use uplift modelling
2. Uplift estimates are noisy because we subtract two models -> the variance adds up
3. When the signal is small relative to the noise, we're just ranking by randomness -> need regularisation
4. Shrinkage pulls unreliable estimates toward zero, so only genuinely strong effects survive -> better targeting decisions